In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND SIMULATION CONFIGURATION (PYTHON)
# ===================================================

from pyspark.sql import functions as F

CATALOG = "semiconplus_portfolio"
GOLD_SCHEMA = "gold"
SIMULATION_SCHEMA = "simulation"

FACT_LOT = f"{CATALOG}.{GOLD_SCHEMA}.fact_lot_performance"
DIM_LOT = f"{CATALOG}.{GOLD_SCHEMA}.dim_lot"
SIMULATED_LANDING = (
    f"{CATALOG}.{SIMULATION_SCHEMA}.simulated_retest_events_landing"
)

SIMULATION_SEED = 20260818
SIMULATION_VERSION = "RETEST_V1"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SIMULATION_SCHEMA}")
spark.conf.set("spark.sql.session.timeZone", "UTC")

print("Day 3 deterministic retest configuration loaded.")


In [0]:
# ===================================================
# BLOCK 2 — SELECT ELIGIBLE FIRST-PASS FAILURES (PYTHON)
# ===================================================

eligible_lots_df = (
    spark.table(FACT_LOT).alias("fact")
    .join(
        spark.table(DIM_LOT).alias("lot"),
        F.col("fact.lot_key") == F.col("lot.lot_key"),
        "inner",
    )
    .filter(F.col("fact.first_pass_fail_quantity") > 0)
    .select(
        F.col("fact.lot_key"),
        F.col("fact.source_lot_id"),
        F.col("fact.input_quantity"),
        F.col("fact.first_pass_good_quantity"),
        F.col("fact.first_pass_fail_quantity"),
        F.col("lot.device_id"),
        F.col("lot.product_group_id"),
        F.col("lot.site_id"),
        F.col("lot.equipment_id"),
        F.col("lot.lot_completion_timestamp_utc"),
    )
)

eligible_count = eligible_lots_df.count()
assert eligible_count > 0, "No first-pass-failure lots are eligible for simulation."
print(f"Eligible first-pass-failure lots: {eligible_count:,}")

In [0]:
# ===================================================
# BLOCK 3 — GENERATE DETERMINISTIC RETEST EVENTS (PYTHON)
# ===================================================

"""
All pseudo-random-looking values come from xxhash64 over immutable business
keys plus a fixed seed. No unseeded rand() function is used.
"""

base_df = (
    eligible_lots_df
    .withColumn(
        "input_basis_points",
        F.pmod(
            F.xxhash64(
                F.lit(SIMULATION_SEED),
                F.col("source_lot_id"),
                F.lit("RETEST_INPUT"),
            ),
            F.lit(5001),
        ) + F.lit(3500),
    )
    .withColumn(
        "good_basis_points",
        F.pmod(
            F.xxhash64(
                F.lit(SIMULATION_SEED),
                F.col("source_lot_id"),
                F.lit("RETEST_GOOD"),
            ),
            F.lit(2501),
        ) + F.lit(7000),
    )
    .withColumn(
        "delay_minutes",
        F.pmod(
            F.xxhash64(
                F.lit(SIMULATION_SEED),
                F.col("source_lot_id"),
                F.lit("RETEST_DELAY"),
            ),
            F.lit(120),
        ) + F.lit(1),
    )
    .withColumn(
        "test_time_milliseconds",
        F.pmod(
            F.xxhash64(
                F.lit(SIMULATION_SEED),
                F.col("source_lot_id"),
                F.lit("RETEST_TIME"),
            ),
            F.lit(30001),
        ) + F.lit(8000),
    )
    .withColumn(
        "defect_index",
        F.pmod(
            F.xxhash64(
                F.lit(SIMULATION_SEED),
                F.col("source_lot_id"),
                F.lit("DEFECT"),
            ),
            F.lit(6),
        ).cast("int"),
    )
)

simulated_retest_df = (
    base_df
    .withColumn(
        "retest_input_quantity",
        F.greatest(
            F.lit(1),
            F.floor(
                F.col("first_pass_fail_quantity")
                * F.col("input_basis_points")
                / F.lit(10000)
            ).cast("long"),
        ),
    )
    .withColumn(
        "retest_good_quantity",
        F.floor(
            F.col("retest_input_quantity")
            * F.col("good_basis_points")
            / F.lit(10000)
        ).cast("long"),
    )
    .withColumn(
        "retest_fail_quantity",
        F.col("retest_input_quantity") - F.col("retest_good_quantity"),
    )
    .withColumn(
        "retest_timestamp_utc",
        F.expr(
            "lot_completion_timestamp_utc + "
            "make_interval(0, 0, 0, 0, 0, delay_minutes, 0)"
        ),
    )
    .withColumn(
        "defect_code",
        F.element_at(
            F.array(
                F.lit("CONTACT"),
                F.lit("FUNCTIONAL"),
                F.lit("HANDLER_JAM"),
                F.lit("LEAKAGE"),
                F.lit("OPEN_SHORT"),
                F.lit("TIMING"),
            ),
            F.col("defect_index") + F.lit(1),
        ),
    )
    .withColumn("error_code", F.concat(F.lit("RT_"), F.col("defect_code")))
    .withColumn(
        "retest_test_time_seconds",
        (F.col("test_time_milliseconds") / F.lit(1000.0)).cast("double"),
    )
    .withColumn(
        "retest_event_id",
        F.sha2(
            F.concat_ws(
                "|",
                F.lit(SIMULATION_VERSION),
                F.lit(str(SIMULATION_SEED)),
                F.col("source_lot_id"),
            ),
            256,
        ),
    )
    .select(
        "retest_event_id",
        "retest_timestamp_utc",
        "source_lot_id",
        "device_id",
        "product_group_id",
        "site_id",
        "equipment_id",
        "defect_code",
        "error_code",
        "retest_input_quantity",
        "retest_good_quantity",
        "retest_fail_quantity",
        "retest_test_time_seconds",
        F.lit(SIMULATION_SEED).cast("long").alias("simulation_seed"),
        F.lit(SIMULATION_VERSION).alias("simulation_version"),
        F.lit(True).alias("simulated_record_flag"),
        F.lit("DETERMINISTIC_PORTFOLIO_SIMULATION").alias("record_origin"),
    )
)

In [0]:
# ===================================================
# BLOCK 4 — VALIDATE BEFORE PUBLISHING (PYTHON)
# ===================================================

validation_df = simulated_retest_df.alias("sim").join(
    eligible_lots_df.select(
        "source_lot_id",
        "first_pass_fail_quantity",
        "first_pass_good_quantity",
        "input_quantity",
    ).alias("eligible"),
    "source_lot_id",
    "inner",
)

invalid_rows = validation_df.filter(
    (F.col("retest_input_quantity") < 0)
    | (F.col("retest_input_quantity") > F.col("first_pass_fail_quantity"))
    | (F.col("retest_good_quantity") < 0)
    | (F.col("retest_good_quantity") > F.col("retest_input_quantity"))
    | (
        F.col("retest_fail_quantity")
        != F.col("retest_input_quantity") - F.col("retest_good_quantity")
    )
    | (
        F.col("first_pass_good_quantity") + F.col("retest_good_quantity")
        > F.col("input_quantity")
    )
    | (~F.col("simulated_record_flag"))
).count()

duplicate_event_ids = (
    simulated_retest_df.groupBy("retest_event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert invalid_rows == 0, f"Invalid simulated retest rows: {invalid_rows}"
assert duplicate_event_ids == 0, "Duplicate retest_event_id values found."
assert simulated_retest_df.count() == eligible_count

print("Pre-publication retest constraints passed.")

In [0]:
# ===================================================
# BLOCK 5 — PUBLISH THE SIMULATED LANDING TABLE (PYTHON)
# ===================================================

published_df = (
    simulated_retest_df
    .withColumn("_simulation_published_at_utc", F.current_timestamp())
    .withColumn("_simulation_source", F.lit("SEMICONPLUS_DAY3_RETEST_V1"))
)

(
    published_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SIMULATED_LANDING)
)

signature_row = (
    simulated_retest_df
    .select(
        F.sha2(
            F.concat_ws(
                "|",
                "retest_event_id",
                "source_lot_id",
                "retest_input_quantity",
                "retest_good_quantity",
                "retest_fail_quantity",
                "simulation_seed",
            ),
            256,
        ).alias("row_signature")
    )
    .agg(
        F.count("*").alias("row_count"),
        F.min("row_signature").alias("minimum_row_signature"),
        F.max("row_signature").alias("maximum_row_signature"),
    )
)

print(f"Simulated landing rows: {spark.table(SIMULATED_LANDING).count():,}")
display(signature_row)
display(spark.table(SIMULATED_LANDING).orderBy("source_lot_id").limit(20))